# 05 - Demo agente LangGraph

Demo local del grafo Hunter + SDR usando filas reales de `analysis/top50.csv`.

In [1]:
from pathlib import Path
import os
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent.resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Cargar .env local sin imprimir secretos.
env_path = ROOT / '.env'
if env_path.exists():
    for line in env_path.read_text(encoding='utf-8').splitlines():
        if not line or line.strip().startswith('#') or '=' not in line:
            continue
        key, value = line.split('=', 1)
        os.environ.setdefault(key.strip(), value.strip())

import pandas as pd
from IPython.display import Markdown, display

from src.agents.graph import compiled_graph
from src.db.models import DEFAULT_DB_PATH

# Reset determinístico de DB local ignorada para evitar duplicados entre ejecuciones del notebook.
for suffix in ['', '-wal', '-shm']:
    db_path = Path(str(DEFAULT_DB_PATH) + suffix)
    if db_path.exists():
        db_path.unlink()

top50 = pd.read_csv(ROOT / 'analysis' / 'top50.csv')
assert os.environ.get('GROQ_API_KEY'), 'Falta GROQ_API_KEY en entorno o .env para clasificar replies reales con Groq'
print('Setup OK: graph importado, DB local reseteada, GROQ_API_KEY cargada sin mostrar valor')

Setup OK: graph importado, DB local reseteada, GROQ_API_KEY cargada sin mostrar valor


## Grafo Mermaid

In [2]:
mermaid = compiled_graph.get_graph().draw_mermaid()
print(mermaid)
display(Markdown('```mermaid\n' + mermaid + '\n```'))

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	nodo_brief_hunter(nodo_brief_hunter)
	nodo_chequeo_duplicado(nodo_chequeo_duplicado)
	nodo_enviar_email(nodo_enviar_email)
	nodo_clasificar_reply(nodo_clasificar_reply)
	nodo_router(nodo_router)
	nodo_handoff_agendar(nodo_handoff_agendar)
	nodo_handoff_nurture(nodo_handoff_nurture)
	nodo_handoff_descarte(nodo_handoff_descarte)
	__end__([<p>__end__</p>]):::last
	__start__ -.-> nodo_brief_hunter;
	__start__ -.-> nodo_chequeo_duplicado;
	nodo_chequeo_duplicado -.-> __end__;
	nodo_chequeo_duplicado -.-> nodo_enviar_email;
	nodo_clasificar_reply --> nodo_router;
	nodo_enviar_email --> nodo_clasificar_reply;
	nodo_router -.-> __end__;
	nodo_router -.-> nodo_handoff_agendar;
	nodo_router -.-> nodo_handoff_descarte;
	nodo_router -.-> nodo_handoff_nurture;
	nodo_brief_hunter --> __end__;
	nodo_handoff_agendar --> __end__;
	nodo_handoff_descarte --> __end__;
	nodo_handoff_nurture --> __end__;
	classD

```mermaid
---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	nodo_brief_hunter(nodo_brief_hunter)
	nodo_chequeo_duplicado(nodo_chequeo_duplicado)
	nodo_enviar_email(nodo_enviar_email)
	nodo_clasificar_reply(nodo_clasificar_reply)
	nodo_router(nodo_router)
	nodo_handoff_agendar(nodo_handoff_agendar)
	nodo_handoff_nurture(nodo_handoff_nurture)
	nodo_handoff_descarte(nodo_handoff_descarte)
	__end__([<p>__end__</p>]):::last
	__start__ -.-> nodo_brief_hunter;
	__start__ -.-> nodo_chequeo_duplicado;
	nodo_chequeo_duplicado -.-> __end__;
	nodo_chequeo_duplicado -.-> nodo_enviar_email;
	nodo_clasificar_reply --> nodo_router;
	nodo_enviar_email --> nodo_clasificar_reply;
	nodo_router -.-> __end__;
	nodo_router -.-> nodo_handoff_agendar;
	nodo_router -.-> nodo_handoff_descarte;
	nodo_router -.-> nodo_handoff_nurture;
	nodo_brief_hunter --> __end__;
	nodo_handoff_agendar --> __end__;
	nodo_handoff_descarte --> __end__;
	nodo_handoff_nurture --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc

```

## Corridas con filas reales

In [3]:
def brand_row(brand_id, **overrides):
    row = top50.loc[top50['brand_id'].eq(brand_id)]
    assert not row.empty, f'No existe {brand_id} en top50.csv'
    data = row.iloc[0].dropna().to_dict()
    data.update(overrides)
    return data

cases = [
    {
        'name': 'Tier A - Hunter Sr',
        'brand_id': 'Brand_0002',
        'state': {
            'brand_data': brand_row('Brand_0002'),
            'tier': 'A',
            'ya_contactado': False,
            'reply_recibido': None,
            'clasificacion': None,
            'decision': '',
            'log_razonamiento': [],
        },
    },
    {
        'name': 'Tier B - Agendar',
        'brand_id': 'Brand_0145',
        'state': {
            'brand_data': brand_row('Brand_0145', contacto_email='demo@example.com'),
            'tier': 'B',
            'ya_contactado': False,
            'reply_recibido': 'Sí me interesa, podemos agendar una llamada esta semana con mi equipo de ecommerce.',
            'clasificacion': None,
            'decision': '',
            'log_razonamiento': [],
        },
    },
    {
        'name': 'Tier B - Opt-out / descarte',
        'brand_id': 'Brand_0826',
        'state': {
            'brand_data': brand_row('Brand_0826', contacto_email='demo@example.com'),
            'tier': 'B',
            'ya_contactado': False,
            'reply_recibido': 'Por favor no me vuelvan a escribir, no me interesa.',
            'clasificacion': None,
            'decision': '',
            'log_razonamiento': [],
        },
    },
]

results = []
for case in cases:
    result = compiled_graph.invoke(case['state'])
    results.append({'case': case['name'], 'brand_id': case['brand_id'], 'result': result})
    print(f"\n=== {case['name']} | {case['brand_id']} ===")
    print(f"decision: {result.get('decision')}")
    print(f"clasificacion: {result.get('clasificacion')}")
    print('log_razonamiento:')
    for line in result.get('log_razonamiento', []):
        print(f"- {line}")


=== Tier A - Hunter Sr | Brand_0002 ===
decision: brief_hunter
clasificacion: None
log_razonamiento:
- nodo_brief_hunter: Tier A detectado; brief enviado/preparado en Slack.
- nodo_brief_hunter: no hay outreach automático para Tier A.



=== Tier B - Agendar | Brand_0145 ===
decision: agendar
clasificacion: {'intent_score': 90, 'is_decision_maker': True, 'objection_type': None, 'suggested_action': 'agendar', 'reasoning': 'El merchant muestra interés claro y propone agendar una llamada con su equipo de ecommerce', 'es_opt_out': False}
log_razonamiento:
- nodo_chequeo_duplicado: sin contacto reciente; continúa outreach.
- nodo_enviar_email: email D0 preparado/enviado a demo@example.com; contactado_en marcado en SQLite.
- nodo_clasificar_reply: suggested_action=agendar | El merchant muestra interés claro y propone agendar una llamada con su equipo de ecommerce
- nodo_router: próxima rama=agendar.
- nodo_handoff_agendar: handoff Hunter preparado en Slack; SLA sugerido <24h.

=== Tier B - Opt-out / descarte | Brand_0826 ===
decision: descartar
clasificacion: {'intent_score': 0, 'is_decision_maker': False, 'objection_type': None, 'suggested_action': 'descartar', 'reasoning': 'Opt-out explícito detectado antes de cualquier a

## Resumen estructurado

In [4]:
summary = pd.DataFrame([
    {
        'case': item['case'],
        'brand_id': item['brand_id'],
        'tier': item['result'].get('tier'),
        'decision': item['result'].get('decision'),
        'log_steps': len(item['result'].get('log_razonamiento', [])),
    }
    for item in results
])
summary

,case,brand_id,tier,decision,log_steps
0,Tier A - Hunter Sr,Brand_0002,A,brief_hunter,2
1,Tier B - Agendar,Brand_0145,B,agendar,5
2,Tier B - Opt-out / descarte,Brand_0826,B,descartar,6
